# Shell Mesh Builder

Build a 3D shell-like mesh by sweeping an aperture along a logarithmic growth path.

Conceptually, this treats the shell as a record of accretionary growth:

* The spiral path represents the route followed by the growing aperture.
* At each point on the path, an elliptical aperture is generated.
* The aperture expands as the shell grows.
* Axial translation, supplied either by z_path or z_scale, lifts the growth path through space and allows high-spired forms.
* Aperture tilt introduces a simple shear, causing one side of the aperture to sit higher than the other. This gives the whorl profile a slight inclination and helps neighbouring whorls read as overlapping shell growth rather than as a separate coiled tube.
* Consecutive apertures are stitched together into a continuous surface.
* Periodic changes in aperture size create growth ribs.
* Late-stage enlargement of the aperture creates a flared shell lip.
* Optional shell wall thickness creates an outer and inner shell surface.
* A colour value is assigned to each vertex to simulate shell banding.

The shell wall thickness is deliberately simple: each aperture ring is duplicated as a slightly smaller inner ring. The outer rings form the external shell surface, while the inner rings form the inner shell wall. The final aperture is stitched between the outer and inner rings so that the shell opening has a visible rim.

## Important implementation detail

* The outer shell vertices are stored first.
* Inner shell vertices are appended afterwards.

This keeps the first part of the mesh compatible with the chamber/septa builder, which uses the original outer aperture rings as the edges for internal chamber walls.

## Ribbing controls

- Smaller ribbing_amplitude gives subtler ribs.
- Larger ribbing_amplitude gives stronger raised growth ribs.
- Smaller ribbing_frequency gives broader, more widely spaced ribs.
- Larger ribbing_frequency gives finer, more closely spaced ribs.

# Lip controls

- lip_start controls when the final aperture flare begins. For example, 0.88 means the lip starts forming after 88% of growth.
- lip_strength controls how much the final aperture expands.
- lip_power controls how gradually or abruptly the flare develops.

# Wall controls

- wall_enabled controls whether an inner shell surface is generated.
- wall_thickness controls how far the inner surface is inset from the outer surface. Smaller values give thinner shell walls.
- inner_surface_colour_shift can darken or lighten the inner shell surface relative to the outer pigmentation.

In [ ]:
import numpy as np

In [ ]:
def growth_fraction(idx, n_spiral):
    """
    Return the normalised growth fraction for a spiral index.

    :param idx: Current growth-path index
    :param n_spiral: Total number of growth-path positions
    :return: Fraction from 0.0 to 1.0
    """
    return idx / (n_spiral - 1)

In [ ]:
def compute_ribbing_factor(theta_value, enabled=True, amplitude=0.08, frequency=10):
    """
    Compute the geometric ribbing multiplier for an aperture ring.

    :param theta_value: Spiral angle at the current growth point
    :param enabled: Whether ribbing is enabled
    :param amplitude: Strength of aperture modulation
    :param frequency: Frequency of ribs along the growth path
    :return: Multiplicative ribbing factor
    """
    if not enabled:
        return 1.0

    return 1 + amplitude * np.sin(frequency * theta_value)

In [ ]:
def compute_lip_factor(idx, n_spiral, enabled=True, lip_start=0.88,
                       lip_strength=0.35, lip_power=3):
    """
    Compute the late-growth aperture flare multiplier.

    :param idx: Current growth-path index
    :param n_spiral: Total number of growth-path positions
    :param enabled: Whether lip flare is enabled
    :param lip_start: Growth fraction at which flare begins
    :param lip_strength: Maximum proportional aperture enlargement
    :param lip_power: Controls gradual versus abrupt flare development
    :return: Multiplicative lip flare factor
    """
    if not enabled:
        return 1.0

    gf = growth_fraction(idx, n_spiral)

    if gf < lip_start:
        return 1.0

    lip_t = (gf - lip_start) / (1 - lip_start)
    return 1 + lip_strength * lip_t**lip_power

In [ ]:
def compute_inner_axes(aw, ah, wall_enabled=True, wall_thickness=0.04):
    """
    Compute the inner aperture semi-axes used for shell wall thickness.

    :param aw: Outer aperture semi-width
    :param ah: Outer aperture semi-height
    :param wall_enabled: Whether an inner wall is generated
    :param wall_thickness: Absolute inset from the outer aperture
    :return: Inner aperture semi-width and semi-height
    """
    if not wall_enabled:
        return aw, ah

    inner_aw = max(aw - wall_thickness, aw * 0.25)
    inner_ah = max(ah - wall_thickness, ah * 0.25)

    return inner_aw, inner_ah

In [ ]:
def pigmentation_band(theta_value, phi_value, ribbing_factor):
    """
    Compute a simple shell pigmentation value for one vertex.

    :param theta_value: Spiral angle at the current growth point
    :param phi_value: Position around the aperture ring
    :param ribbing_factor: Ribbing multiplier for the current aperture
    :return: Scalar colour/intensity value
    """
    return np.sin(8 * theta_value) + 0.4 * np.sin(3 * phi_value) * ribbing_factor

In [ ]:
def aperture_point(cx, cy, cz, local_x, local_z, angle, aperture_tilt):
    """
    Convert local aperture coordinates into world-space shell coordinates.

    :param cx: Aperture centre x coordinate
    :param cy: Aperture centre y coordinate
    :param cz: Aperture centre z coordinate
    :param local_x: Local horizontal aperture coordinate
    :param local_z: Local vertical aperture coordinate
    :param angle: Rotation angle of the aperture in the x/y plane
    :param aperture_tilt: Shear applied to z according to local_x
    :return: World-space x, y, z coordinates
    """
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)

    px = cx + local_x * cos_a
    py = cy + local_x * sin_a
    pz = cz + local_z + aperture_tilt * local_x

    return px, py, pz

In [ ]:
def build_aperture_rings(
    theta, x, y, aperture_width, aperture_height,
    z_scale=0.08,
    z_path=None,
    aperture_points=32,
    aperture_tilt=0.25,
    ribbing_enabled=True,
    ribbing_amplitude=0.08,
    ribbing_frequency=10,
    lip_enabled=True,
    lip_start=0.88,
    lip_strength=0.35,
    lip_power=3,
    wall_enabled=True,
    wall_thickness=0.04,
    inner_surface_colour_shift=-0.15
):
    """
    Build outer and optional inner aperture-ring vertices along the growth path.

    :param theta: Spiral angle values
    :param x: Spiral x coordinates
    :param y: Spiral y coordinates
    :param aperture_width: Aperture width at each growth point
    :param aperture_height: Aperture height at each growth point
    :param z_scale: Vertical rise factor if z_path is not supplied
    :param z_path: Optional explicit z coordinates for the growth path
    :param aperture_points: Number of vertices around each aperture ring
    :param aperture_tilt: Shear applied to incline the aperture
    :param ribbing_enabled: Whether geometric ribbing is enabled
    :param ribbing_amplitude: Strength of ribbing modulation
    :param ribbing_frequency: Rib frequency along the shell
    :param lip_enabled: Whether final aperture flare is enabled
    :param lip_start: Growth fraction where lip flare begins
    :param lip_strength: Maximum proportional lip enlargement
    :param lip_power: Shape exponent for lip flare
    :param wall_enabled: Whether inner shell wall vertices are generated
    :param wall_thickness: Inset used for inner shell wall
    :param inner_surface_colour_shift: Colour shift applied to inner wall
    :return: Outer vertex arrays and inner vertex arrays
    """
    n_spiral = len(theta)
    phi_values = np.linspace(0, 2*np.pi, aperture_points, endpoint=False)

    X_outer, Y_outer, Z_outer, C_outer = [], [], [], []
    X_inner, Y_inner, Z_inner, C_inner = [], [], [], []

    for idx in range(n_spiral):
        cx = x[idx]
        cy = y[idx]
        cz = z_path[idx] if z_path is not None else z_scale * theta[idx]

        aw = aperture_width[idx] / 2
        ah = aperture_height[idx] / 2

        rib_factor = compute_ribbing_factor(
            theta[idx],
            enabled=ribbing_enabled,
            amplitude=ribbing_amplitude,
            frequency=ribbing_frequency
        )

        lip_factor = compute_lip_factor(
            idx,
            n_spiral,
            enabled=lip_enabled,
            lip_start=lip_start,
            lip_strength=lip_strength,
            lip_power=lip_power
        )

        aw *= rib_factor * lip_factor
        ah *= rib_factor * lip_factor

        inner_aw, inner_ah = compute_inner_axes(
            aw,
            ah,
            wall_enabled=wall_enabled,
            wall_thickness=wall_thickness
        )

        for p in phi_values:
            band = pigmentation_band(theta[idx], p, rib_factor)

            outer_local_x = aw * np.cos(p)
            outer_local_z = ah * np.sin(p)

            ox, oy, oz = aperture_point(
                cx, cy, cz,
                outer_local_x,
                outer_local_z,
                theta[idx],
                aperture_tilt
            )

            X_outer.append(ox)
            Y_outer.append(oy)
            Z_outer.append(oz)
            C_outer.append(band)

            if wall_enabled:
                inner_local_x = inner_aw * np.cos(p)
                inner_local_z = inner_ah * np.sin(p)

                ix, iy, iz = aperture_point(
                    cx, cy, cz,
                    inner_local_x,
                    inner_local_z,
                    theta[idx],
                    aperture_tilt
                )

                X_inner.append(ix)
                Y_inner.append(iy)
                Z_inner.append(iz)
                C_inner.append(band + inner_surface_colour_shift)

    return X_outer, Y_outer, Z_outer, C_outer, X_inner, Y_inner, Z_inner, C_inner

In [ ]:
def stitch_ring_surface(I, J, K, ring_offset, n_spiral, n_aperture, reverse=False):
    """
    Stitch neighbouring aperture rings into a continuous triangular surface.

    :param I: Triangle index list I
    :param J: Triangle index list J
    :param K: Triangle index list K
    :param ring_offset: Vertex offset for the first ring
    :param n_spiral: Number of aperture rings
    :param n_aperture: Number of vertices per aperture ring
    :param reverse: Whether to reverse triangle winding
    """
    for s in range(n_spiral - 1):
        for p in range(n_aperture):
            current = ring_offset + s * n_aperture + p
            next_p = ring_offset + s * n_aperture + ((p + 1) % n_aperture)

            current_next = ring_offset + (s + 1) * n_aperture + p
            next_next = ring_offset + (s + 1) * n_aperture + ((p + 1) % n_aperture)

            if not reverse:
                I.extend([current, next_p])
                J.extend([current_next, current_next])
                K.extend([next_p, next_next])
            else:
                I.extend([current, next_p])
                J.extend([next_p, next_next])
                K.extend([current_next, current_next])

In [ ]:
def stitch_wall_rim(I, J, K, outer_start, inner_start, n_aperture, reverse=False):
    """
    Stitch an outer aperture ring to an inner aperture ring to form a rim.

    :param I: Triangle index list I
    :param J: Triangle index list J
    :param K: Triangle index list K
    :param outer_start: Start index of the outer aperture ring
    :param inner_start: Start index of the inner aperture ring
    :param n_aperture: Number of vertices in each aperture ring
    :param reverse: Whether to reverse the triangle winding
    """
    for p in range(n_aperture):
        outer_current = outer_start + p
        outer_next = outer_start + ((p + 1) % n_aperture)

        inner_current = inner_start + p
        inner_next = inner_start + ((p + 1) % n_aperture)

        if not reverse:
            I.extend([outer_current, outer_next])
            J.extend([inner_current, inner_current])
            K.extend([outer_next, inner_next])
        else:
            I.extend([outer_current, outer_next])
            J.extend([outer_next, inner_next])
            K.extend([inner_current, inner_current])

In [ ]:
def build_shell_mesh(
    theta,
    r,
    x,
    y,
    aperture_width,
    aperture_height,
    z_scale=0.08,
    z_path=None,
    aperture_points=32,
    aperture_tilt=0.25,
    ribbing_enabled=True,
    ribbing_amplitude=0.08,
    ribbing_frequency=10,
    lip_enabled=True,
    lip_start=0.88,
    lip_strength=0.35,
    lip_power=3,
    wall_enabled=True,
    wall_thickness=0.04,
    inner_surface_colour_shift=-0.15
):
    """
    Build a 3D shell mesh by sweeping an aperture along a growth path.

    :param theta: Spiral angle values representing growth progression
    :param r: Spiral radius values; retained for API compatibility
    :param x: Spiral x coordinates
    :param y: Spiral y coordinates
    :param aperture_width: Width of elliptical aperture at each growth point
    :param aperture_height: Height of elliptical aperture at each growth point
    :param z_scale: Vertical rise factor if z_path is not supplied
    :param z_path: Optional explicit z coordinates for the growth path
    :param aperture_points: Number of vertices around each aperture ring
    :param aperture_tilt: Shear applied to incline the aperture
    :param ribbing_enabled: Whether to add geometric growth ribs
    :param ribbing_amplitude: Strength of ribbing modulation
    :param ribbing_frequency: Frequency of ribs along the shell
    :param lip_enabled: Whether to flare the final aperture
    :param lip_start: Fraction of growth at which lip flare begins
    :param lip_strength: Maximum proportional lip enlargement
    :param lip_power: Shape exponent for lip flare
    :param wall_enabled: Whether to generate an inner shell wall
    :param wall_thickness: Inset used for inner wall thickness
    :param inner_surface_colour_shift: Colour shift for the inner wall
    :return: X, Y, Z, C vertex arrays and triangle index arrays I, J, K
    """
    n_spiral = len(theta)
    n_aperture = aperture_points

    (
        X_outer, Y_outer, Z_outer, C_outer,
        X_inner, Y_inner, Z_inner, C_inner
    ) = build_aperture_rings(
        theta, x, y,
        aperture_width,
        aperture_height,
        z_scale=z_scale,
        z_path=z_path,
        aperture_points=aperture_points,
        aperture_tilt=aperture_tilt,
        ribbing_enabled=ribbing_enabled,
        ribbing_amplitude=ribbing_amplitude,
        ribbing_frequency=ribbing_frequency,
        lip_enabled=lip_enabled,
        lip_start=lip_start,
        lip_strength=lip_strength,
        lip_power=lip_power,
        wall_enabled=wall_enabled,
        wall_thickness=wall_thickness,
        inner_surface_colour_shift=inner_surface_colour_shift
    )

    if wall_enabled:
        X = np.array(X_outer + X_inner)
        Y = np.array(Y_outer + Y_inner)
        Z = np.array(Z_outer + Z_inner)
        C = np.array(C_outer + C_inner)
    else:
        X = np.array(X_outer)
        Y = np.array(Y_outer)
        Z = np.array(Z_outer)
        C = np.array(C_outer)

    I, J, K = [], [], []

    stitch_ring_surface(
        I, J, K,
        ring_offset=0,
        n_spiral=n_spiral,
        n_aperture=n_aperture,
        reverse=False
    )

    if wall_enabled:
        inner_offset = n_spiral * n_aperture

        stitch_ring_surface(
            I, J, K,
            ring_offset=inner_offset,
            n_spiral=n_spiral,
            n_aperture=n_aperture,
            reverse=True
        )

        final_outer_start = (n_spiral - 1) * n_aperture
        final_inner_start = inner_offset + (n_spiral - 1) * n_aperture

        stitch_wall_rim(
            I, J, K,
            outer_start=final_outer_start,
            inner_start=final_inner_start,
            n_aperture=n_aperture,
            reverse=False
        )

        initial_outer_start = 0
        initial_inner_start = inner_offset

        stitch_wall_rim(
            I, J, K,
            outer_start=initial_outer_start,
            inner_start=initial_inner_start,
            n_aperture=n_aperture,
            reverse=True
        )

    return X, Y, Z, C, I, J, K